In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
GEPA_RUN  = "32B_H200"
GEPA_PATH = Path(f"GEPA_runs/{GEPA_RUN}")

In [3]:
# %run ../gs_slimming.py
# print("="*120)
# print("gs_slimming done.")
# print("="*120)
%run GEPA_runs/flattening.py -p "{GEPA_PATH}"
print("="*120)
print("flattening done.")
print("="*120)


  Flattening 33 JSONs => GEPA_runs/32B_H200/0.csv
  Reports    : 33
  Zeilen     : 337
  Fehler     : 0
  Gespeichert: GEPA_runs/32B_H200/0.csv

  Flattening 32 JSONs => GEPA_runs/32B_H200/1.csv
  Reports    : 32
  Zeilen     : 368
  Fehler     : 0
  Gespeichert: GEPA_runs/32B_H200/1.csv

  Flattening 32 JSONs => GEPA_runs/32B_H200/10.csv
  Reports    : 32
  Zeilen     : 290
  Fehler     : 0
  Gespeichert: GEPA_runs/32B_H200/10.csv

  Flattening 32 JSONs => GEPA_runs/32B_H200/11.csv
  Reports    : 32
  Zeilen     : 310
  Fehler     : 0
  Gespeichert: GEPA_runs/32B_H200/11.csv

  Flattening 32 JSONs => GEPA_runs/32B_H200/12.csv
  Reports    : 31
  Zeilen     : 446
  Fehler     : 0
  Gespeichert: GEPA_runs/32B_H200/12.csv

  Flattening 32 JSONs => GEPA_runs/32B_H200/13.csv
  Reports    : 31
  Zeilen     : 361
  Fehler     : 0
  Gespeichert: GEPA_runs/32B_H200/13.csv

  Flattening 32 JSONs => GEPA_runs/32B_H200/14.csv
  Reports    : 32
  Zeilen     : 320
  Fehler     : 0
  Gespeichert: G

## Import slimmed down Gold-Standard

In [4]:
gs = pd.read_json("../gs_slim.json")
gs["year"] = gs["year"].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import all Extractions

In [5]:
CSV_LIST = sorted(
    [p for p in GEPA_PATH.glob("*.csv") if p.stem.isdigit()],
    key=lambda p: int(p.stem)
)

oa = {i: pd.read_csv(CSV_LIST[i]) for i in range(len(CSV_LIST))}

df_to_merge = [(oa[i], f"_{i}") for i in range(len(CSV_LIST))]
## this makes the same as
# df_to_merge = [
#     (think1, "_t1"),
#     (think2, "_t2"),
#     (moe1, "_m1"),
#     (moe2, "_m2"),
#     (instr1, "_i1"),
#     (instr2, "_i2")
# ]
## but for the many GEPA iterations (up to 30 at the moment)

In [6]:
extracted_report_names = set()
for df, _ in df_to_merge:
    extracted_report_names.update(df["report_name"].unique())

gs = gs[gs["report_name"].isin(extracted_report_names)].reset_index(drop=True)

print(f"{len(extracted_report_names)} extracted reports:")
print(sorted(extracted_report_names))

33 extracted reports:
['Allianz_2022_report', 'Fresenius SE_2019_report', 'ViacomCBS_ESG Report_2020-2021_vFINAL', 'acuity brands inc_2022_report', 'addtech_2022_report', 'aixtron_2020_report', 'autoneum holding_2019_report', 'bekaert (d) sa_2022_report', 'blackline safety_2021_report', 'cabot corp_2018_report', 'dampskibsselskabet norden_2019_report', 'evolution mining ltd_2020_report', 'fujifilm_2022_report', 'georg fischer ag_2018_report', 'granite construction inc_2020_report', 'h. lundbeck_2021_report', 'hammerson reit_2021_report', 'huhtamaki_2018_report', 'inchcape plc_2022_report', 'innospec inc_2020_report', 'jetblue airways corp_2019_report', 'kfw_2018_report', 'kilroy realty_2017_report', 'kilroy realty_2018_report', 'morinaga ltd_2020_report', 'nok corporation_2021_report', 'nordea_2017_report', 'salzgitter ag_2018_report', 'sato oyj_2022_report', 'stantec inc_2019_report', 'toshiba tec corp_2022_report', 'uniper_2019_report', 'varta ag_2021_report']


In [7]:
# Year normalization RegEx-Script

import re

def normalize_year(raw: str, years_present: set[int] | None = None) -> tuple[int | None, str]:
    """Map a raw fiscal-year label to a calendar year. Returns (year, rule_applied)."""
    
    if isinstance(raw, float):
        if pd.isna(raw):
            return None, "unparseable"
        raw = int(raw) if raw.is_integer() else raw
    
    label = str(raw).strip().upper().removeprefix("FY").strip()

    if re.fullmatch(r"\d{4}", label):
        return int(label), "plain"
    if re.fullmatch(r"\d{2}", label):
        return 2000 + int(label), "fy_2digit"

    m = re.fullmatch(r"(\d{4})/(\d{1,4})", label)
    if m:
        left, right = m.groups()
        if len(right) == 4:
            return int(left), "range_former_year"  # 2021/2022 -> 2022
        candidates = {int(left), int(left) + 1}
        if years_present:
            hit = candidates & years_present
            if len(hit) == 1:
                return hit.pop(), "fy_end_month_context"
        return int(left), "fy_end_month_fallback"

    return None, "unparseable"

In [8]:
# Get years from extractions 
years_in_extractions = set()
for df, _ in df_to_merge:
    years_in_extractions.update(df["year"].unique().tolist())

# Create ynrom DataFrames to preserve the original ones
oa_copies = {i: oa[i].copy() for i in range(len(CSV_LIST))}

df_to_merge_ynorm = [(oa_copies[i], f"_{i}") for i in range(len(CSV_LIST))]

# Now normalization via the RegEx Script from above
for df, _ in df_to_merge_ynorm:
    df["year"], df["year_rule"] = zip(*df["year"].apply(
        normalize_year,
        years_present=years_in_extractions
    ))

## Matching Extractions to Gold-Standard Scheme

### After ynrom

In [9]:
years = set()
year_report = []

for df, _ in df_to_merge_ynorm:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements
    
print(sorted(years))

[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, 2014, nan, 2016, 2017, 2018, 2019, 2020, 2021, 2022, nan, 2015, nan, nan, nan]


## Mapping special occurences

In [10]:
scope_mapping = {
    "scope_1": "1",
    "scope_2_location_based": "2lb",
    "scope_2_market_based" : "2mb",
    "scope_3" : "3"
}

# Prints out only if sth. goes wrong
for df, sf in df_to_merge:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



Total NaN nach Konversion: 0


In [11]:

# Prints out only if sth. goes wrong
for df, sf in df_to_merge_ynorm:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



Total NaN nach Konversion: 0


Checking if all `report_names` are the same

And checking if all `report_names` from the extractions are exactly matched in the GoldStandard

## Merging Extraction Values and Gold-Standard

In [12]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

In [13]:
merged_ynorm = gs.copy()
merged_ynorm["year"] = merged_ynorm["year"].astype(int)

for df, sf in df_to_merge_ynorm:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged_ynorm = pd.merge(merged_ynorm, agg, on=merge_on, how="left")

merged_ynorm.info()

<class 'pandas.DataFrame'>
RangeIndex: 1362 entries, 0 to 1361
Data columns (total 86 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      1362 non-null   str    
 1   year             1362 non-null   int64  
 2   scope            1362 non-null   str    
 3   page             333 non-null    float64
 4   value            332 non-null    float64
 5   unit             331 non-null    str    
 6   unit_normalized  331 non-null    str    
 7   label            332 non-null    str    
 8   status           1362 non-null   str    
 9   scopes_present   1362 non-null   object 
 10  years_present    1362 non-null   object 
 11  value_0          345 non-null    object 
 12  unit_0           345 non-null    object 
 13  label_0          345 non-null    object 
 14  value_1          320 non-null    object 
 15  unit_1           320 non-null    object 
 16  label_1          320 non-null    object 
 17  value_2          323 non-

#### Rows where no value is present must be from Type "NaN" for better analysis; not "None" because the object is missing.

In [14]:
pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

# for col in pipeline_cols:
#     merged[col] = merged[col].apply(
#     lambda x: np.nan if x is None else x
# )
    
for col in pipeline_cols:
    merged_ynorm[col] = merged_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

### Rearranging Columns

In [15]:

gs_cols = [
    'report_name', 'status', 'scopes_present', 'years_present', # Per-Report granularity
    'year', 'scope',                                            # Per category
    'page', 'value', 'unit',                                    # About the value as-is in the report
    'unit_normalized', 'label',                                 # Additional information
]

pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge_ynorm] +
    [f"unit{s}"  for _, s in df_to_merge_ynorm] +
    [f"label{s}" for _, s in df_to_merge_ynorm]
)

# merged = merged[gs_cols + pipeline_cols]
merged_ynorm = merged_ynorm[gs_cols + pipeline_cols]

merged_ynorm.info()

<class 'pandas.DataFrame'>
RangeIndex: 1362 entries, 0 to 1361
Data columns (total 86 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      1362 non-null   str    
 1   status           1362 non-null   str    
 2   scopes_present   1362 non-null   object 
 3   years_present    1362 non-null   object 
 4   year             1362 non-null   int64  
 5   scope            1362 non-null   str    
 6   page             333 non-null    float64
 7   value            332 non-null    float64
 8   unit             331 non-null    str    
 9   unit_normalized  331 non-null    str    
 10  label            332 non-null    str    
 11  value_0          345 non-null    object 
 12  value_1          320 non-null    object 
 13  value_2          323 non-null    object 
 14  value_3          314 non-null    object 
 15  value_4          306 non-null    object 
 16  value_5          306 non-null    object 
 17  value_6          300 non-

## Saving created DataFrame

In [16]:
# merged.to_json(f"{GEPA_RUN}.json", index=False, orient="records", indent=4)
merged_ynorm.to_json(f"{GEPA_RUN}_ynorm.json", index=False, orient="records", indent=4)

## To later read back the files:
# pd.read_csv("gs_extractions_raw.csv")   # Lists as Strings. Needs ast.literal_eval
# pd.read_json("gs_extractions_raw.json", orient="records")  # Lists instant usable
